# Optimizing Agentic AI Collaboration v3.0.0 — L1 Smoke Test: Self-Consistency

**Purpose:** validate the full SC pipeline — the `utils.persistence`/`utils.voting`/`utils.graphs` machinery that the 8 real SC notebooks (`v3.1.x`–`v3.3.x`) depend on — end-to-end, before committing GPU-hours to real 5×100/5×128 runs. This is the gate for everything downstream (SC → Majority Voting → BoN-MAV).

**Hardware target:** Nvidia T4 GPU (Kaggle / Colab free tier). Runs against the `dev-agentic-baselines` W&B project, never the real results project.

## Grid

**4 models × 2 benchmarks × 2 problems × 5 generators = 80 generations**, using `SMOKE_DATA_SEED=4242` (deliberately different from the real `789`, so this run never touches the real eval subsets) and the full production `GEN` params (unchanged, so parsing/extraction bugs surface). Estimated **1–1.5 h** on a T4 including four model load/unload cycles.

| Model | MATH-500 max_new_tokens | MMLU max_new_tokens |
|---|---|---|
| Ministral-3-3B-Instruct | 1500 | 1024 |
| Qwen3-1.7B | 1500 (`/no_think`) | 1024 (`/no_think`) |
| Qwen3-4B | 1500 (`/no_think`) | 1024 (`/no_think`) |
| Phi-4-mini-instruct | 1500 | 1024 |

## Relation with the project WPs

**WP2 - T4: L1 multi-agent protocol — Self-Consistency.** This notebook is the validation gate for the whole L1 campaign's generator-compute machinery: the persistence artifact schema, the LangGraph voting template, and the SC@k ladder logic, all exercised on real (small) GPU generations rather than mocked.

---
### Before running (Kaggle)

0. Upload this notebook to a Kaggle kernel (or open it directly if already there).
1. In **Settings**, enable **Accelerator: GPU T4 x2** (the notebook pins itself to one GPU).
2. Add the following secrets via **Add-ons → Secrets**, enabling each for the notebook:
   - `GITHUB_TOKEN` — GitHub PAT with repo read scope
   - `HF_TOKEN` — Hugging Face read-access token
   - `WANDB_API_KEY` — Weights & Biases API key
3. Click **Run All**. The first cell auto-detects Kaggle, clones the repo, and loads secrets.

### Before running (Colab)

0. **Copy this notebook** → File → Save a copy in Drive
1. **Enable GPU** → Runtime → Change runtime type → T4 GPU → Save
2. **Add Colab Secrets** (🔑 icon in the left sidebar):
   - `HF_TOKEN` — your Hugging Face read-access token ([create one here](https://huggingface.co/settings/tokens))
   - `WANDB_API_KEY` — your Weights & Biases API key ([find it here](https://wandb.ai/authorize))
   - `GITHUB_TOKEN` — a GitHub Personal Access Token with `repo` read scope
   - **Enable "Notebook access"** for all secrets

## What this smoke test validates

* The **generation pass** (live LangGraph, one agent, per-seed LLM rebuild) for both MATH-500 and MMLU — including the Qwen3 `/no_think` template variant. Grouping and voting are unified across benchmarks via `utils.voting.MATH500_SPEC`/`MMLU_SPEC` (a `BenchmarkSpec` bound once per adapter), and the vote is produced **by the graph's `aggregate` node itself**, not by a Python loop that merely reuses its data.
* **Full-text artifact persistence** (`utils.persistence.log_generations_artifact`) across a resumable per-seed logging loop, and **retrieval** (`load_generations_artifact` via `use_artifact`) in a separate downstream run — proving real W&B producer→consumer lineage, not just in-memory reuse.
* **Alignment validation** (`utils.persistence.validate_alignment`) against the locally loaded smoke subsets, for all 8 producer artifacts.
* The full **SC@1/3/4/5 ladder** (`utils.voting.run_self_consistency` + `compose_subset_metrics`) computed once per (model, benchmark) and logged as 4 separate aggregation runs — the same code path the real notebooks use.
* **Sequential model loading**: exactly one model resident in VRAM at any time, confirmed via `torch.cuda.memory_allocated()` after each unload.
* **Determinism**: re-running `run_self_consistency` on an identical pool from two independent, freshly seeded `random.Random(TIE_BREAK_SEED)` instances yields bit-identical results.

Passing this notebook end-to-end (all cells, all asserts) is the gate before running any of the 8 real `v3.1.x`–`v3.3.x` SC notebooks.

## 1. Setup and Installation

We install the libraries required by the repo (Hugging Face stack, `langchain*`, `langgraph`, `datasets`, `wandb`, `math-verify`; `mistral-common` for the Ministral-3 tokenizer). If running on Colab/Kaggle, the repo (with the `utils` helper modules) is cloned first.

In [ ]:
# ── Environment detection: clone private repo ────────────────
import sys, os

if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    ENVIRONMENT = "kaggle"
elif "COLAB_RELEASE_TAG" in os.environ or "COLAB_BACKEND_VERSION" in os.environ:
    ENVIRONMENT = "colab"
else:
    ENVIRONMENT = "local"


if ENVIRONMENT == "colab":
    repo_root = "/content/tfg-agentic-ai"
    if not os.path.isdir(repo_root):
        try:
            try:
                from google.colab import userdata  # type: ignore
                os.environ.setdefault("GITHUB_TOKEN", userdata.get("GITHUB_TOKEN") or "")
            except ImportError:
                from dotenv import load_dotenv  # type: ignore
                load_dotenv()

            repo_url = f"https://{os.environ.get('GITHUB_TOKEN')}@github.com/andreufc/tfg-agentic-ai.git"
            import subprocess  # type: ignore
            subprocess.run(["git", "clone", "--depth", "1", repo_url, repo_root], check=True)
            print(f"✅ Repo cloned to {repo_root}")
        except Exception as e:
            print(f"Error cloning repo: {e}")
    else:
        print(f"ℹ️  Repo already present at {repo_root}")
    os.chdir(os.path.join(repo_root, "notebooks"))

elif ENVIRONMENT == "kaggle":

    # Kaggle provides 2x T4 GPUs; restrict to a single GPU so that
    # experiments run under the same resource constraints as Colab.
    # This MUST happen before any torch / CUDA import.
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    repo_root = "/kaggle/working/tfg-agentic-ai"
    if not os.path.isdir(repo_root):
        try:
            try:
                from kaggle_secrets import UserSecretsClient  # type: ignore
                _secrets = UserSecretsClient()
                os.environ.setdefault("GITHUB_TOKEN", _secrets.get_secret("GITHUB_TOKEN") or "")
            except ImportError:
                from dotenv import load_dotenv  # type: ignore
                load_dotenv()
            from kaggle_secrets import UserSecretsClient  # type: ignore
            repo_url = f"https://{os.environ.get('GITHUB_TOKEN')}@github.com/andreufc/tfg-agentic-ai.git"
            import subprocess  # type: ignore
            subprocess.run(["git", "clone", "--depth", "1", repo_url, repo_root], check=True)
            print(f"✅ Repo cloned to {repo_root}")
        except Exception as e:
            print(f"Error cloning repo: {e}")
    else:
        print(f"ℹ️  Repo already present at {repo_root}")
    os.chdir(os.path.join(repo_root, "notebooks"))

else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"repo_root = {repo_root}")

In [ ]:
# Only necessary if running in Google Colab, but won't cause issues if run locally
%cd {repo_root}
%pip install -r requirements.txt

## 2. Authentication via Colab / Kaggle Secrets

We pull `HF_TOKEN` and `WANDB_API_KEY` from Colab userdata or Kaggle Secrets — no tokens in code.

In [ ]:
from utils.config import load_env

print(f"Runtime environment: {ENVIRONMENT}")

load_env(environment=ENVIRONMENT)

## 3. Smoke Configuration

The 4 target models are looked up from `utils.registry.MODELS` — the single source of truth for `model_id`/`run_name`/`no_think` shared with every production notebook, not a separate list duplicated here. `SMOKE_DATA_SEED=4242` and `SMOKE_EVAL_SIZE=2` keep this run tiny and disjoint from the real `data_seed=789` subsets. `SC_SEEDS`/`TIE_BREAK_SEED`/`K_LADDER` match production exactly — this is what makes the smoke test a real validation of the production code path, not a simplified stand-in.

In [ ]:
from utils.registry import MODELS
from utils.persistence import SCHEMA_VERSION

PROJECT = "dev-agentic-baselines"   # smoke runs NEVER go to the real results project
print(f"W&B Project: {PROJECT}")

# The 4 target models, looked up from the single registry (utils.registry.MODELS)
# shared with every production notebook — no separate model list duplicated here.
MODEL_KEYS = list(MODELS)

# Same sampling params as production (T=0.7, top_p=0.95, top_k=40 per Wang et al.,
# Self-Consistency) — unchanged in smoke mode so parsing/extraction bugs surface.
GEN_COMMON = {"temperature": 0.7, "top_p": 0.95, "top_k": 40}

# Deliberately different from the real data_seed=789 -> this run never touches the
# real eval subsets. eval_size=2 keeps 4 models x 2 benchmarks x 5 seeds tractable.
SMOKE_DATA_SEED = 4242
SMOKE_EVAL_SIZE = 2

SC_SEEDS       = [42, 123, 456, 654, 987]
TIE_BREAK_SEED = 2026
K_LADDER       = [1, 3, 4, 5]

print(f"✅ {len(MODEL_KEYS)} models x 2 benchmarks x {SMOKE_EVAL_SIZE} problems x {len(SC_SEEDS)} generators "
      f"= {len(MODEL_KEYS) * 2 * SMOKE_EVAL_SIZE * len(SC_SEEDS)} generations planned.")

## 4. Data Loading

Both datasets are loaded once and shared across all 4 models. The 2-problem smoke subsets are drawn with `SMOKE_DATA_SEED=4242` — the same subsets every model in this notebook evaluates against.

In [ ]:
import random
from utils.data import load_math500, load_mmlu
from utils.persistence import hash_task

math500_dataset = load_math500(split="test")
mmlu_dataset    = load_mmlu(split="test")

random.seed(SMOKE_DATA_SEED)
math_smoke_sample = random.sample(list(math500_dataset), SMOKE_EVAL_SIZE)
random.seed(SMOKE_DATA_SEED)
mmlu_smoke_sample = random.sample(list(mmlu_dataset), SMOKE_EVAL_SIZE)

print(f"✅ MATH-500: {len(math500_dataset)} loaded, {len(math_smoke_sample)} sampled for smoke.")
for i, item in enumerate(math_smoke_sample):
    print(f"  [{i}] hash={hash_task(item['problem'])}  {item['problem'][:70]!r}")

print(f"\n✅ MMLU: {len(mmlu_dataset)} loaded, {len(mmlu_smoke_sample)} sampled for smoke.")
for i, item in enumerate(mmlu_smoke_sample):
    print(f"  [{i}] hash={hash_task(item['question'])}  {item['question'][:70]!r}")

## 5. Templates and the Instrumented Generation Closure

`build_templates` produces the MATH-500 and MMLU chain templates for a given model — identical wording to the production notebooks, with the `/no_think` prefix added only for Qwen3 models. `make_generate_fn` is the same instrumented closure builder every production SC notebook uses.

In [ ]:
import time
import wandb
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from utils.models import load_model_and_tokenizer
from utils.instrumentation import timed_invoke, tokens_per_second


def build_templates(no_think: bool) -> dict:
    """MATH-500 and MMLU chain templates, identical wording to production —
    Qwen3 models additionally get the /no_think prefix (thinking mode off)."""
    prefix = "/no_think\n\n" if no_think else ""
    math_template = PromptTemplate.from_template(
        prefix + "Problem:\n{task}\n\n"
        "Please reason step by step, and put your final answer within \\boxed{{}}."
    )
    mmlu_template = PromptTemplate.from_template(
        prefix + "Your answer should be of the following format: 'ANSWER: [LETTER]' "
        "(without quotes) where [LETTER] is one of A,B,C,D. "
        "Think step by step before answering.\n\n{task}"
    )
    return {"math500": math_template, "mmlu": mmlu_template}


def make_generate_fn(chain, prompt_template, model_key, model_id, agent_id, gen_id, extract_fn, tokenizer):
    """Build a live ``generate(task, seed) -> AgentSample`` closure — identical
    to the one every production SC notebook defines, via ``timed_invoke``.
    ``tokenizer`` is passed in explicitly (rather than captured from a
    module-level global) because this notebook loads a different tokenizer per
    model in a loop."""
    def generate(task, seed):
        r = timed_invoke(chain, prompt_template, tokenizer, {"task": task})
        return {
            "agent_id": agent_id, "model_key": model_key, "model_id": model_id, "source": "live",
            "gen_id": gen_id, "seed": seed,
            "answer": r["output_text"], "parsed_output": extract_fn(r["output_text"]),
            "latency_s": r["latency_s"], "input_tokens": r["input_tokens"],
            "output_tokens": r["output_tokens"], "total_tokens": r["total_tokens"],
            "gpu_peak_mb": r["gpu_peak_mb"],
        }
    return generate

print("✅ Template builder and generation closure builder ready.")

## 6. Dataset Adapters

To drive one shared pipeline over both benchmarks without duplicating it 4×2 times (as the production notebooks intentionally do, for per-notebook independence), this smoke test factors the benchmark-specific pieces into two small adapter dicts. Extraction/equivalence/scoring are **not** duplicated here — each adapter just points at the shared `utils.voting.MATH500_SPEC`/`MMLU_SPEC`, exactly like the production notebooks' `SPEC`. The adapter only carries what's genuinely per-benchmark and not already owned by `BenchmarkSpec`: `max_new_tokens` and the W&B metric prefix. Task rendering and problem identity are no longer adapter concerns at all — `spec.render_task` builds the task text and the loop's own ordinal `problem_id` is the identity, exactly like the production notebooks. Everything downstream (`utils.graphs`, `utils.voting`, `utils.persistence`) is the exact same production code.

In [ ]:
from utils.voting import MATH500_SPEC, MMLU_SPEC

# Adapters now carry only what's genuinely notebook-level (per-benchmark
# max_new_tokens and the W&B metric prefix) — task rendering AND problem
# identity are both fully owned by BenchmarkSpec.render_task and the loop's
# own ordinal problem_id, so there is nothing benchmark-specific left to
# adapt for either one.
MATH_ADAPTER = {
    "spec": MATH500_SPEC,
    "max_new_tokens": 1500,
    "metric_prefix": "math_sample",
}

MMLU_ADAPTER = {
    "spec": MMLU_SPEC,
    "max_new_tokens": 1024,
    "metric_prefix": "mmlu_sample",
}

print("✅ MATH-500 and MMLU adapters ready.")

## 7. Pipeline Functions

Two functions, each mirroring one section of the production notebooks exactly:

* `run_generation_pass` — the live LangGraph generation loop (5 seeds × `SMOKE_EVAL_SIZE` problems), per-seed artifact re-logging, generator table, `finish_wandb()`. Returns the in-memory records and the final producer artifact ref.
* `run_sc_k_aggregation` — one W&B run per `k ∈ {1,3,4,5}`, `use_artifact` lineage, reads each k's subset results off `utils.voting.run_self_consistency`'s output (called once in the main loop, per model×benchmark) + `compose_subset_metrics`, `accuracy_statistics`.

Grouping the 5 generations per problem is no longer a separate step here — `run_self_consistency` does it internally, once per problem, regardless of how many `k` values are requested.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import math
import numpy as np
from utils.agents import build_hf_langchain_llm
from utils.graphs import AgentSpec, build_majority_voting_graph
from utils.registry import generations_artifact_name
from utils.persistence import (
    build_generation_store, make_generation_record, artifact_ref, SCHEMA_VERSION,
    log_generations_artifact, load_generations_artifact,
    generation_table_rows, validate_alignment, AlignmentError,
)
from utils.voting import compose_subset_metrics, run_self_consistency
from utils.statistics import accuracy_statistics
from utils.logging_utils import init_wandb, define_wandb_chart_only_metrics, log_metrics, log_stat_summary, finish_wandb


def run_generation_pass(model, tokenizer, model_key, model_cfg, dataset_key, dataset_sample, template, adapter):
    """Live generation pass for one (model, benchmark): 5 seeds x len(dataset_sample)
    problems, through a single-live-agent LangGraph, with per-seed resumable
    artifact logging — identical structure to the production notebooks' cell 10."""
    model_id  = model_cfg["model_id"]
    run_name  = model_cfg["run_name"]
    spec = adapter["spec"]
    max_new_tokens = adapter["max_new_tokens"]
    metric_prefix  = adapter["metric_prefix"]
    gen_config = {"max_new_tokens": max_new_tokens, **GEN_COMMON}

    run_config = {
        "model_key": model_key, "model_id": model_id, "quantization": "nf4-4bit",
        "framework": "langchain+langgraph", "protocol": "self-consistency",
        "eval_size": len(dataset_sample), "data_seed": SMOKE_DATA_SEED,
        "generation_seeds": SC_SEEDS, "tie_break_seed": TIE_BREAK_SEED, "k_ladder": K_LADDER,
        "schema_version": SCHEMA_VERSION,
        **gen_config, "eval_dataset": dataset_key,
    }
    run = init_wandb(
        run_name=f"l1-sc-gen-{run_name}-{dataset_key}-smoke", project=PROJECT, config=run_config,
        tags=["l1", "self-consistency", "generation", dataset_key, "smoke"],
    )
    define_wandb_chart_only_metrics(prefixes=[f"{metric_prefix}/*"])

    # A single agent never ties (nothing to vote against), so this RNG never
    # actually draws — it exists only to satisfy build_majority_voting_graph's
    # signature, exactly like the production notebooks' gen_tie_rng.
    gen_tie_rng = random.Random(TIE_BREAK_SEED)

    artifact_name = generations_artifact_name(model_key, dataset_key) + "-smoke"
    generation_records = []
    producer_ref = None

    for gen_id, seed in enumerate(SC_SEEDS):
        llm = build_hf_langchain_llm(
            model=model, tokenizer=tokenizer, max_new_tokens=max_new_tokens,
            temperature=gen_config["temperature"], top_p=gen_config["top_p"], top_k=gen_config["top_k"],
            return_full_text=False, seed=seed,
        )
        chain = template | llm | StrOutputParser()
        generate_fn = make_generate_fn(chain, template, model_key, model_id, "gen", gen_id, spec.extract, tokenizer)
        agent_spec = AgentSpec(agent_id="gen", model_key=model_key, model_id=model_id, mode="live", generate=generate_fn, seed=seed)
        graph = build_majority_voting_graph([agent_spec], spec, rng=gen_tie_rng)

        for problem_id, item in enumerate(dataset_sample):
            task  = spec.render_task(item)
            truth = spec.get_truth(item)

            out = graph.invoke({
                "task": task, "problem_id": problem_id,
                "dataset": dataset_key, "samples": [], "vote": None, "metrics": None,
            })
            sample = out["samples"][0]
            compare_value = sample["answer"] if spec.compare_on == "answer" else sample["parsed_output"]
            correct = spec.score(compare_value, truth)
            tps     = tokens_per_second(sample["output_tokens"], sample["latency_s"])

            print(f"  ⏱  {model_key}/{dataset_key} gen_id={gen_id} problem={problem_id}: "
                  f"{sample['latency_s']:.1f}s  correct={correct}")

            generation_records.append(make_generation_record(
                dataset=dataset_key, problem_id=problem_id, model_key=model_key, gen_id=gen_id,
                seed=seed, protocol="self-consistency", task=task, sample=sample,
                correct=correct, tokens_per_s=tps,
            ))
            log_metrics({f"{metric_prefix}/accuracy": int(correct)})

        manifest = {
            "schema_version": SCHEMA_VERSION, "model_key": model_key, "dataset": dataset_key,
            "protocol": "self-consistency", "data_seed": SMOKE_DATA_SEED, "eval_size": len(dataset_sample),
            "generation_seeds": SC_SEEDS[:gen_id + 1], "gen_params": gen_config,
            "wandb_run_id": run.id, "wandb_run_name": run.name, "wandb_project": PROJECT,
            "n_records": len(generation_records),
            "created_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }
        artifact = log_generations_artifact(generation_records, manifest, name=artifact_name, run=run)
        # Captured off the actually-committed version (artifact.wait()'d inside
        # log_generations_artifact) — the round-trip validation cell downloads
        # exactly this pinned ref, not a computed ":vN" guess.
        producer_ref = artifact_ref(artifact, entity=run.entity, project=PROJECT)

    columns, rows = generation_table_rows(generation_records)
    wandb.log({f"generations_{dataset_key}": wandb.Table(columns=columns, data=rows)})
    finish_wandb()

    return generation_records, producer_ref


def run_sc_k_aggregation(model_key, model_cfg, dataset_key, producer_ref, store, sc_all, dataset_sample, adapter):
    """One W&B run per k in K_LADDER — use_artifact lineage, reads each problem's
    subset results off sc_all (utils.voting.run_self_consistency's output, computed
    once in the main loop) + compose_subset_metrics, accuracy_statistics — the same
    functions the production SC@k cell calls."""
    model_id = model_cfg["model_id"]
    run_name = model_cfg["run_name"]
    n_generators = len(SC_SEEDS)
    sample_size  = len(dataset_sample)
    metric_prefix = adapter["metric_prefix"]

    k_results = {}
    for k in K_LADDER:
        n_subsets = math.comb(n_generators, k)

        sc_run = init_wandb(
            run_name=f"l1-sc{k}-{run_name}-{dataset_key}-smoke", project=PROJECT,
            config={
                "model_key": model_key, "model_id": model_id, "protocol": "self-consistency",
                "eval_size": sample_size, "data_seed": SMOKE_DATA_SEED, "generation_seeds": SC_SEEDS,
                "tie_break_seed": TIE_BREAK_SEED, "sc_k": k, "n_subsets": n_subsets,
                "producer_artifact": producer_ref, "eval_dataset": dataset_key,
            },
            tags=["l1", "self-consistency", f"sc{k}", dataset_key, "smoke"],
        )
        sc_run.use_artifact(producer_ref)
        define_wandb_chart_only_metrics(prefixes=[f"{metric_prefix}/*"])

        correctness_matrix = np.zeros((sample_size, n_subsets), dtype=int)
        time_matrix        = np.zeros((sample_size, n_subsets), dtype=float)
        gpu_peak_matrix     = np.zeros((sample_size, n_subsets), dtype=float)
        n_tie = 0
        n_abstain_total = 0

        for problem_id in range(sample_size):
            subset_results = sc_all[problem_id]["by_k"][k]
            records = [store[(problem_id, g)] for g in range(n_generators)]

            for subset_idx, sr in enumerate(subset_results):
                metrics = compose_subset_metrics([records[i] for i in sr["indices"]])
                correctness_matrix[problem_id, subset_idx] = int(sr["correct"])
                time_matrix[problem_id, subset_idx]        = metrics["latency_s"]
                gpu_peak_matrix[problem_id, subset_idx]     = metrics["gpu_peak_mb"]
                if sr["tie"]:
                    n_tie += 1
                n_abstain_total += sr["n_abstain"]
                log_metrics({f"{metric_prefix}/accuracy": int(sr["correct"])})

        acc_stats = accuracy_statistics(correctness_matrix)
        wandb.summary["accuracy"]     = round(acc_stats["p_hat"], 6)
        wandb.summary["n_subsets"]    = n_subsets
        wandb.summary["tie_rate"]     = round(n_tie / (sample_size * n_subsets), 4)
        wandb.summary["abstain_rate"] = round(n_abstain_total / (sample_size * n_subsets * n_generators), 4)
        log_stat_summary("accuracy_stats", acc_stats, keys=["ci_lower", "ci_upper"])
        finish_wandb()

        k_results[k] = {
            "accuracy": acc_stats["p_hat"], "n_subsets": n_subsets,
            "correctness_matrix": correctness_matrix, "gpu_peak_matrix": gpu_peak_matrix,
        }
        print(f"  ✅ {model_key}/{dataset_key} SC@{k}: accuracy={acc_stats['p_hat']:.3f} "
              f"(n_subsets={n_subsets})")

    return k_results

print("✅ Pipeline functions (run_generation_pass, run_sc_k_aggregation) ready.")

## 8. Main Loop — 4 Models × 2 Benchmarks

For each model: load → warmup → for each benchmark: run the generation pass, then `utils.voting.run_self_consistency` (one fresh `random.Random(TIE_BREAK_SEED)` per benchmark, mirroring the production notebooks' Vote Once cell) → the SC@k aggregation runs → unload and assert VRAM is released before the next model. Exactly one model is ever resident, matching the production single-model-per-notebook VRAM budget.

In [ ]:
import gc
import torch
from utils.warmup import warmup_model

producer_refs      = {}   # (model_key, dataset_key) -> artifact ref
in_memory_records  = {}   # (model_key, dataset_key) -> generation_records
sc_results         = {}   # (model_key, dataset_key) -> {k: {...}}

DATASETS = [
    ("math500", math_smoke_sample, MATH_ADAPTER),
    ("mmlu", mmlu_smoke_sample, MMLU_ADAPTER),
]

for model_key in MODEL_KEYS:
    model_cfg = MODELS[model_key]
    print(f"\n{'#'*70}\n  MODEL: {model_cfg['model_id']}\n{'#'*70}")

    model, tokenizer = load_model_and_tokenizer(model_cfg["model_id"], quantize_4bit=True)
    print(f"✅ Model loaded on {model.device} — dtype {model.dtype}")

    warmup_model(model, tokenizer, max_new_tokens=64, n_iterations=1,
                 temperature=GEN_COMMON["temperature"], top_p=GEN_COMMON["top_p"], top_k=GEN_COMMON["top_k"])

    templates = build_templates(no_think=model_cfg["no_think"])

    for dataset_key, dataset_sample, adapter in DATASETS:
        print(f"\n{'-'*70}\n  {model_key} / {dataset_key}\n{'-'*70}")

        records, producer_ref = run_generation_pass(
            model, tokenizer, model_key, model_cfg, dataset_key, dataset_sample, templates[dataset_key], adapter,
        )
        key = (model_key, dataset_key)
        in_memory_records[key] = records
        producer_refs[key] = producer_ref

        # Fresh, independently seeded RNG — reproducible on its own, mirroring the
        # production notebooks' Vote Once cell.
        store = build_generation_store(records)
        vote_tie_rng = random.Random(TIE_BREAK_SEED)
        sc_all = run_self_consistency(
            store, dataset_sample, adapter["spec"], K_LADDER,
            n_generators=len(SC_SEEDS), rng=vote_tie_rng,
            model_key=model_key, model_id=model_cfg["model_id"],
        )
        k_results = run_sc_k_aggregation(model_key, model_cfg, dataset_key, producer_ref, store, sc_all, dataset_sample, adapter)
        sc_results[key] = k_results

    # ── Unload before the next model — assert VRAM actually drops ──────────
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    allocated_mb = torch.cuda.memory_allocated() / 1e6
    print(f"\nVRAM after unloading {model_key}: {allocated_mb:.1f} MB")
    assert allocated_mb < 1000, (
        f"VRAM not released after unloading {model_cfg['model_id']}: {allocated_mb:.1f} MB"
    )

print(f"\n{'='*70}\n  MAIN LOOP COMPLETE — {len(in_memory_records)} (model, dataset) pairs generated.\n{'='*70}")

## 9. Reuse Round-Trip Validation

With **no model loaded**, download all 8 producer artifacts via `use_artifact` (real W&B lineage, explicit `:v0`-through-final version refs already captured in `producer_refs`), rebuild each store, and validate alignment against the locally loaded smoke subsets — exactly the workflow Majority Voting and BoN-MAV notebooks will use against the real artifacts.

In [ ]:
print(f"\n{'='*70}\n  REUSE ROUND-TRIP VALIDATION (no model loaded)\n{'='*70}")

round_trip_run = init_wandb(
    run_name="l1-sc-reuse-roundtrip-smoke", project=PROJECT,
    config={"purpose": "round-trip validation", "producer_refs": {f"{k[0]}/{k[1]}": v for k, v in producer_refs.items()}},
    tags=["l1", "self-consistency", "smoke", "roundtrip"],
)

downloaded_stores = {}
downloaded_manifests = {}
alignment_errors = []

for model_key in MODEL_KEYS:
    for dataset_key, dataset_sample, adapter in DATASETS:
        key = (model_key, dataset_key)
        ref = producer_refs[key]

        records, manifest = load_generations_artifact(ref, run=round_trip_run)
        store = build_generation_store(records)
        downloaded_stores[key] = store
        downloaded_manifests[key] = manifest

        try:
            validate_alignment(
                manifest, store, dataset_sample, render_task=adapter["spec"].render_task,
                expected_dataset=dataset_key, expected_model_key=model_key,
                expected_data_seed=SMOKE_DATA_SEED, expected_eval_size=SMOKE_EVAL_SIZE,
                n_generators=len(SC_SEEDS),
            )
            print(f"  ✅ alignment OK: {key[0]}/{key[1]}  ({ref})")
        except AlignmentError as e:
            alignment_errors.append((key, str(e)))
            print(f"  ❌ alignment FAILED: {key[0]}/{key[1]}: {e}")

assert not alignment_errors, f"Alignment failures: {alignment_errors}"

# Byte-compare fidelity: a downloaded record's full text vs the in-memory original.
fidelity_key = (MODEL_KEYS[0], "math500")
fidelity_store = downloaded_stores[fidelity_key]
fidelity_original = in_memory_records[fidelity_key][0]
fidelity_downloaded = fidelity_store[(fidelity_original["problem_id"], fidelity_original["gen_id"])]
assert fidelity_downloaded["answer"] == fidelity_original["answer"], (
    "Artifact round-trip fidelity FAILED: `answer` text mismatch after download."
)
print(f"\n✅ Artifact round-trip fidelity confirmed: downloaded `answer` is byte-identical "
      f"to the in-memory original ({len(fidelity_original['answer'])} chars).")

finish_wandb()

## 10. Invariant Checklist

Hard asserts covering every claim this smoke test needs to establish before the 8 real SC notebooks run.

In [ ]:
print(f"\n{'='*70}\n  INVARIANT CHECKLIST\n{'='*70}\n")

from utils.persistence import assert_unique_pk

checks = []

# (a) generation count == 4 x 2 x 2 x 5 = 80
total_generations = sum(len(recs) for recs in in_memory_records.values())
expected_total = len(MODEL_KEYS) * len(DATASETS) * SMOKE_EVAL_SIZE * len(SC_SEEDS)
checks.append((
    f"generation count == {len(MODEL_KEYS)}x{len(DATASETS)}x{SMOKE_EVAL_SIZE}x{len(SC_SEEDS)} = {expected_total}",
    total_generations == expected_total, f"got {total_generations}",
))

# (b) SC@k ladder present for every (model, benchmark) with correct subset counts, SC@5 in [0,1]
expected_subset_counts = {1: 5, 3: 10, 4: 5, 5: 1}
ladder_ok = True
for key, k_results in sc_results.items():
    for k, expected_n in expected_subset_counts.items():
        if k not in k_results or k_results[k]["n_subsets"] != expected_n:
            ladder_ok = False
        elif not (0.0 <= k_results[k]["accuracy"] <= 1.0):
            ladder_ok = False
checks.append(("SC@k ladder present for every (model, benchmark) with subset counts 5/10/5/1, SC@5 in [0,1]",
                ladder_ok, ""))

# (c) artifact round-trip fidelity (re-asserted from the previous cell)
checks.append(("artifact round-trip fidelity (byte-identical `answer` after download)",
                fidelity_downloaded["answer"] == fidelity_original["answer"], ""))

# (d) alignment checks passed for all 8 producer artifacts
checks.append(("alignment checks passed for all 8 producer artifacts", not alignment_errors, str(alignment_errors)))

# (e) primary-key uniqueness across the merged 8-artifact record set — the
# on-Kaggle proof that (dataset, problem_id, model_key, gen_id) holds as a real
# PK once every model x benchmark's records are pooled together.
all_records = [r for recs in in_memory_records.values() for r in recs]
pk_error = None
try:
    assert_unique_pk(all_records)
except ValueError as e:
    pk_error = str(e)
checks.append(("primary key (dataset, problem_id, model_key, gen_id) unique across all 8 artifacts",
                pk_error is None, pk_error or ""))

# (f)+(g) determinism and metric composition — built directly off build_generation_store,
# NOT sliced from the raw records list, whose append order is gen_id-major (all of
# gen_id=0's problems, then gen_id=1's, ...), not problem_id-major.
det_model_key = MODEL_KEYS[0]
det_key = (det_model_key, "math500")
store_det = build_generation_store(in_memory_records[det_key])

# Two independent, freshly seeded runs of the exact production entry point must
# agree bit-for-bit — this is the reproducibility property the whole SC ladder
# relies on for offline re-runs (see utils.voting.run_self_consistency).
r1 = run_self_consistency(store_det, math_smoke_sample, MATH_ADAPTER["spec"], [5],
                          n_generators=len(SC_SEEDS), rng=random.Random(TIE_BREAK_SEED),
                          model_key=det_model_key, model_id=MODELS[det_model_key]["model_id"])
r2 = run_self_consistency(store_det, math_smoke_sample, MATH_ADAPTER["spec"], [5],
                          n_generators=len(SC_SEEDS), rng=random.Random(TIE_BREAK_SEED),
                          model_key=det_model_key, model_id=MODELS[det_model_key]["model_id"])
checks.append(("vote determinism (identical run_self_consistency results from a fresh seed)", r1 == r2, ""))

subset_recs = [store_det[(0, g)] for g in range(len(SC_SEEDS))]  # problem_id=0, gen_id=0..4
manual_latency = sum(r["latency_s"] for r in subset_recs)
manual_peak    = max(r["gpu_peak_mb"] for r in subset_recs)
composed = compose_subset_metrics(subset_recs)
checks.append((
    "metric composition matches manual sums/max",
    abs(composed["latency_s"] - manual_latency) < 1e-6 and abs(composed["gpu_peak_mb"] - manual_peak) < 1e-6,
    f"composed={composed['latency_s']:.2f}/{composed['gpu_peak_mb']:.1f} vs "
    f"manual={manual_latency:.2f}/{manual_peak:.1f}",
))

# (h) VRAM bounds + release between models
all_peaks = [r["gpu_peak_mb"] for recs in in_memory_records.values() for r in recs]
final_allocated_mb = torch.cuda.memory_allocated() / 1e6
checks.append(("every recorded gpu_peak_mb < 16000 (16 GB budget)",
                all(p < 16000 for p in all_peaks), f"max={max(all_peaks):.0f} MB"))
checks.append(("VRAM released after the last model unload",
                final_allocated_mb < 1000, f"{final_allocated_mb:.1f} MB"))

# (i) lineage recorded: producer_ref captured for all 8 (model, dataset) pairs
checks.append(("lineage: producer_ref recorded for all 8 (model, dataset) pairs",
                len(producer_refs) == len(MODEL_KEYS) * len(DATASETS) and all(producer_refs.values()), ""))

# (j) extraction-failure / tie rate diagnostics (printed, not asserted — informational)
n_records_all = sum(len(recs) for recs in in_memory_records.values())
n_extraction_failures = sum(
    1 for recs in in_memory_records.values() for r in recs if r["parsed_output"] is None
)
extraction_failure_rate = n_extraction_failures / n_records_all if n_records_all else 0.0

print(f"Extraction-failure rate across all {n_records_all} generations: {extraction_failure_rate:.2%}\n")

all_passed = True
for name, passed, detail in checks:
    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"  {status}  {name}" + (f"   ({detail})" if detail else ""))
    all_passed = all_passed and passed

assert all_passed, "One or more invariant checks FAILED — see the table above."

print(f"\n{'='*70}\n  ALL INVARIANT CHECKS PASSED ✅\n{'='*70}")

## 11. Smoke Test Complete

If every cell above ran without an assertion failure, this run has validated:

- Full-text generation persistence and W&B artifact retrieval (`utils.persistence`) with real producer→consumer lineage.
- The LangGraph majority-voting template (`utils.graphs`) in live mode, for both MATH-500 and MMLU, including the Qwen3 `/no_think` path — the vote itself produced by the graph's `aggregate` node, via the shared `BenchmarkSpec` (`utils.voting.MATH500_SPEC`/`MMLU_SPEC`).
- The SC@k ladder (`utils.voting.run_self_consistency`, `compose_subset_metrics`) and its statistics/summary logging, matching 2.0.0 conventions.
- The primary key `(dataset, problem_id, model_key, gen_id)` is genuinely unique once every model x benchmark's records are pooled together — the on-Kaggle proof (`utils.persistence.assert_unique_pk`) that the readable-ID scheme holds under real data, not just synthetic fixtures.
- Sequential single-model VRAM discipline across 4 models.
- Deterministic, reproducible voting: two independent runs from a freshly seeded `random.Random(TIE_BREAK_SEED)` agree bit-for-bit.

**Next step:** paste this notebook's W&B run URLs (generation runs, SC@k runs, and the round-trip validation run — all tagged `smoke`) into a new `docs/claims-l1.md` under "Smoke tests", then proceed to the 8 real SC notebooks (`v3.1.0`–`v3.3.1`).